In [1]:
# !pip install python-telegram-bot --upgrade

In [2]:
# !pip uninstall telegram

In [4]:
import os 
from dotenv import load_dotenv

In [5]:
env_path = r"D:\common_credentials\.env"
load_dotenv(dotenv_path=env_path)
# llm= ChatGroq(model='llama-3.1-8b-instant')

True

In [6]:

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
TELEGRAM_TOKEN = os.getenv("TELEGRAM_TOKEN")


# os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
# os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
# os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_TRACING_V2"]="true"

In [15]:
from telegram.ext import Application
from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters, ContextTypes
import google.generativeai as genai
from telegram import Update

In [11]:
# Configure the generative model
genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

In [12]:
model.generate_content("Hi")

response:
GenerateContentResponse(
    done=True,
    iterator=None,
    result=protos.GenerateContentResponse({
      "candidates": [
        {
          "content": {
            "parts": [
              {
                "text": "Hi there! How can I help you today?\n"
              }
            ],
            "role": "model"
          },
          "finish_reason": "STOP",
          "avg_logprobs": -0.00038474539972164416
        }
      ],
      "usage_metadata": {
        "prompt_token_count": 1,
        "candidates_token_count": 11,
        "total_token_count": 12
      },
      "model_version": "gemini-1.5-flash"
    }),
)

In [14]:
def generate_content(full_prompt):
    try:
        response= model.generate_content(full_prompt)
        return response.text if hasattr(response, "text") else "Sorry, I can't generate a response"
    except Exception as e:
        return "There was error while generating the output.."

generate_content("Hi")

'Hi there! How can I help you today?\n'

In [16]:
async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    user_id = update.effective_user.first_name
    system_message= f"Hello {user_id}! I am chabot. How can I assist you today?"
    await  update.message.reply_text(system_message)

In [17]:
async def chat(update: Update, context: ContextTypes.DEFAULT_TYPE) -> None:
    user_message = update.message.text
    response_text = generate_content(user_message)
    await update.message.reply_text(response_text)

In [18]:
# Build the application
app = ApplicationBuilder().token(TELEGRAM_TOKEN).build()

In [19]:
# Add handlers
app.add_handler(CommandHandler("start", start))
app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, chat))

In [20]:
app.run_polling()

RuntimeError: Cannot close a running event loop